# XGBoost v002 — Baseline · L1 · L2 · L1+L2 규제 실험

팀 프로젝트용 Jupyter Notebook. **v002 train/validation**을 사용해 거시지표 포함/제외 × 규제 4종(총 8개)을 비교합니다. 최종 테스트는 사용하지 않습니다.

> **검증 제한:** 예측 origin별로 그때까지의 **train 데이터 정답만** 학습합니다. 2022–2025 validation 정답을 이후 origin 학습에 추가하지 않으므로 완전한 expanding-origin 검증은 아닙니다. 규제 비교를 위한 동일 조건의 train-only 검증입니다.

## STEP 1. 라이브러리 · 경로 · 실험 설정
`ROOT`가 프로젝트 폴더인지 확인하세요. 기본값은 `D:\hjs\five-stack`입니다.

In [1]:
from pathlib import Path
import json
import platform

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

ROOT = Path(r"D:\hjs\five-stack")
DATA = ROOT / "data" / "shared" / "v002"
OUT = ROOT / "artifacts" / "xgboost" / "v002" / "regularization"
SEED = 42
MACRO = ["gdp_usd", "gdp_per_capita_usd", "population", "inflation_pct",
         "exchange_rate_lcu_per_usd", "internet_users_pct"]
BASE_PARAMS = dict(n_estimators=250, max_depth=5, learning_rate=0.05,
                   subsample=0.9, colsample_bytree=0.9,
                   objective="reg:squarederror", tree_method="hist",
                   random_state=SEED, n_jobs=-1)
REGULARIZATIONS = {
    "baseline": {"reg_alpha": 0.0, "reg_lambda": 1.0},
    "l1": {"reg_alpha": 1.0, "reg_lambda": 1.0},
    "l2": {"reg_alpha": 0.0, "reg_lambda": 5.0},
    "l1_l2": {"reg_alpha": 1.0, "reg_lambda": 5.0},
}

print("DATA:", DATA)
print("OUT:", OUT)

DATA: D:\hjs\five-stack\data\shared\v002
OUT: D:\hjs\five-stack\artifacts\xgboost\v002\regularization


## STEP 2. 함수 정의
전처리(결측치 대체, 범주형 One-Hot Encoding), XGBoost, WAPE/MAE/RMSE/sMAPE 계산 함수입니다. 숫자형 StandardScaler는 적용하지 않습니다.

In [2]:
def quarter_number(series: pd.Series) -> pd.Series:
    s = series.astype("string")
    return (pd.to_numeric(s.str[:4], errors="raise").astype(int) * 4
            + pd.to_numeric(s.str[-1], errors="raise").astype(int) - 1)

def make_pipeline(columns, categorical, params):
    numeric = [c for c in columns if c not in categorical]
    preprocessor = ColumnTransformer([
        ("cat", Pipeline([
            ("fill", SimpleImputer(strategy="constant", fill_value="__MISSING__")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical),
        ("num", Pipeline([
            ("fill", SimpleImputer(strategy="constant", fill_value=0,
                                   keep_empty_features=True)),
        ]), numeric),
    ])
    return Pipeline([("preprocessor", preprocessor),
                     ("model", XGBRegressor(**params))])

def inputs(frame, columns, categorical):
    x = frame[columns].copy()
    for c in categorical:
        x[c] = x[c].astype("string").fillna("__MISSING__").astype(str)
    for c in columns:
        if c not in categorical:
            x[c] = pd.to_numeric(x[c], errors="coerce").astype(float)
    return x

def metrics(frame):
    y = frame["actual_usd"].to_numpy(dtype=float)
    p = frame["predicted_usd"].to_numpy(dtype=float)
    e = np.abs(y - p)
    denom = np.abs(y).sum()
    sm_den = np.abs(y) + np.abs(p)
    return {
        "rows": len(frame),
        "WAPE_pct": 100 * e.sum() / denom if denom else np.nan,
        "MAE_usd": float(e.mean()),
        "RMSE_usd": float(np.sqrt(np.mean((y - p) ** 2))),
        "sMAPE_pct": float(100 * np.mean(np.divide(
            2 * e, sm_den, out=np.zeros_like(e), where=sm_den != 0))),
    }

def make_metrics_table(predictions):
    grouped = []
    for group_by in [
        ["model_variant"],
        ["model_variant", "horizon_quarters"],
        ["model_variant", "country_code"],
        ["model_variant", "hs_code"],
    ]:
        for key, subset in predictions.groupby(group_by, dropna=False):
            if not isinstance(key, tuple):
                key = (key,)
            row = dict(zip(group_by, key))
            row.update(metrics(subset))
            row["grouping"] = "+".join(group_by)
            grouped.append(row)
    return pd.DataFrame(grouped)

## STEP 3. v002 데이터 읽기 및 검사
Manifest에 등록된 특성만 사용하고, 시간 순서와 금지 특성을 확인합니다.

In [3]:
manifest = json.loads((DATA / "manifest.json").read_text(encoding="utf-8"))
if manifest.get("dataset_version") != "v002":
    raise ValueError("Expected v002 manifest")
train = pd.read_parquet(DATA / "train.parquet")
val = pd.read_parquet(DATA / "validation.parquet")
features = list(manifest["feature_columns"])
target = manifest["feature_contract"]["target"]
categorical = list(manifest["feature_contract"]["categorical"])
forbidden = set(manifest["feature_contract"]["forbidden_feature_columns"])
if set(features) & forbidden:
    raise ValueError("Forbidden features included in training")
if not set(categorical).issubset(features):
    raise ValueError("Categorical features missing from feature list")
for frame in (train, val):
    frame["_target_qn"] = quarter_number(frame["target_quarter"])
    frame["_origin_qn"] = quarter_number(frame["origin_quarter"])
    frame[target] = pd.to_numeric(frame[target], errors="coerce")
if not train["target_quarter"].between("2013Q1", "2021Q4").all():
    raise ValueError("Unexpected train target period")
if not val["target_quarter"].between("2022Q1", "2025Q4").all():
    raise ValueError("Unexpected validation target period")
if not (val["_origin_qn"] < val["_target_qn"]).all():
    raise ValueError("Origin must precede target quarter")

print("버전:", manifest["dataset_version"], "Train:", train.shape, "Validation:", val.shape)

버전: v002 Train: (28800, 89) Validation: (12800, 89)


## STEP 4. 실험 변수 구성
**no_macro:** 거시지표 6개와 각각의 `_missing` 6개 제외 (기존 기준 45개). **with_macro:** 전체 57개.

In [4]:
macro_exclude = set(MACRO + [f"{m}_missing" for m in MACRO])
variants = {
    "no_macro": [c for c in features if c not in macro_exclude],
    "with_macro": features,
}
if len(variants["no_macro"]) != 45 or len(features) != 57:
    print("WARNING: feature counts differ from previously observed 45/57")
OUT.mkdir(parents=True, exist_ok=True)
print("Dataset:", manifest["dataset_version"], "Train:", train.shape,
      "Validation:", val.shape, flush=True)

print({name:len(cols) for name,cols in variants.items()})

Dataset: v002 Train: (28800, 89) Validation: (12800, 89)
{'no_macro': 45, 'with_macro': 57}


## STEP 5. 8개 실험 학습 · 검증 · 모델 저장
**가장 오래 걸리는 셀**입니다. 4개 규제 설정 × 거시지표 2개 × origin별 모델 재학습을 수행합니다. 각 조합의 train-only 재사용 모델도 `.joblib`로 저장합니다.

- Baseline: L1=0, L2=1
- L1: L1=1, L2=1
- L2 강화: L1=0, L2=5
- L1+L2: L1=1, L2=5

In [5]:
predictions_all = []
origins = sorted(val["origin_quarter"].dropna().unique(),
                 key=lambda q: int(q[:4]) * 4 + int(q[-1]))
for reg_name, reg_params in REGULARIZATIONS.items():
    params = {**BASE_PARAMS, **reg_params}
    for macro_name, cols in variants.items():
        name = f"{reg_name}_{macro_name}"
        print(f"\nRunning {name} | {len(cols)} features | "
              f"alpha={reg_params['reg_alpha']} lambda={reg_params['reg_lambda']}",
              flush=True)
        for origin in origins:
            origin_qn = int(quarter_number(pd.Series([origin])).iloc[0])
            fit = train.loc[
                (train["_target_qn"] <= origin_qn)
                & train[target].notna()
                & train["target_available"].fillna(False)
            ]
            score = val.loc[
                (val["origin_quarter"] == origin)
                & val[target].notna()
                & val["target_available"].fillna(False)
            ]
            if fit.empty or score.empty:
                continue
            pipeline = make_pipeline(cols, categorical, params)
            pipeline.fit(inputs(fit, cols, categorical),
                         fit[target].to_numpy(dtype=float))
            pred = np.maximum(pipeline.predict(inputs(score, cols, categorical)), 0)
            result = score[["country_code", "hs_code", "origin_quarter",
                            "target_quarter", "horizon_quarters"]].copy()
            result["actual_usd"] = score[target].to_numpy(dtype=float)
            result["predicted_usd"] = pred
            result["model_variant"] = name
            result["reg_alpha"] = reg_params["reg_alpha"]
            result["reg_lambda"] = reg_params["reg_lambda"]
            result["fit_rows"] = len(fit)
            predictions_all.append(result)
            print(f"  {origin}: train={len(fit)} validation={len(score)}", flush=True)

        full_fit = train.loc[train[target].notna()
                             & train["target_available"].fillna(False)]
        final_model = make_pipeline(cols, categorical, params)
        final_model.fit(inputs(full_fit, cols, categorical),
                        full_fit[target].to_numpy(dtype=float))
        joblib.dump({
            "pipeline": final_model,
            "feature_columns": cols,
            "categorical_columns": categorical,
            "dataset_version": "v002",
            "model_variant": name,
            "parameters": params,
            "trained_through_target_quarter": "2021Q4",
            "note": "Train-only reusable model; validation-origin models are not saved",
        }, OUT / f"{name}_train_only.joblib")
        print("Saved model:", name, flush=True)



Running baseline_no_macro | 45 features | alpha=0.0 lambda=1.0
  2020Q4: train=20565 validation=137
  2021Q1: train=21225 validation=284
  2021Q2: train=21910 validation=425
  2021Q3: train=22605 validation=565
  2021Q4: train=23290 validation=703
  2022Q1: train=23290 validation=706
  2022Q2: train=23290 validation=704
  2022Q3: train=23290 validation=703
  2022Q4: train=23290 validation=705
  2023Q1: train=23290 validation=715
  2023Q2: train=23290 validation=718
  2023Q3: train=23290 validation=719
  2023Q4: train=23290 validation=723
  2024Q1: train=23290 validation=727
  2024Q2: train=23290 validation=728
  2024Q3: train=23290 validation=730
  2024Q4: train=23290 validation=584
  2025Q1: train=23290 validation=440
  2025Q2: train=23290 validation=294
  2025Q3: train=23290 validation=145
Saved model: baseline_no_macro

Running baseline_with_macro | 57 features | alpha=0.0 lambda=1.0
  2020Q4: train=20565 validation=137
  2021Q1: train=21225 validation=284
  2021Q2: train=21910 val

## STEP 6. 평가 지표 및 CSV 저장
전체 성능뿐 아니라 horizon/국가/제품군별 WAPE를 저장합니다. **WAPE가 낮을수록 좋습니다.**

In [6]:
if not predictions_all:
    raise RuntimeError("No validation predictions")
predictions = pd.concat(predictions_all, ignore_index=True)
predictions.to_csv(OUT / "validation_predictions.csv", index=False,
                   encoding="utf-8-sig")
metrics_df = make_metrics_table(predictions)
metrics_df.to_csv(OUT / "metrics.csv", index=False, encoding="utf-8-sig")
summary = (metrics_df.loc[metrics_df["grouping"] == "model_variant",
                          ["model_variant", "rows", "WAPE_pct", "MAE_usd",
                           "RMSE_usd", "sMAPE_pct"]]
           .sort_values("WAPE_pct"))
summary.to_csv(OUT / "metrics_summary.csv", index=False, encoding="utf-8-sig")
display(summary.reset_index(drop=True))

,model_variant,rows,WAPE_pct,MAE_usd,RMSE_usd,sMAPE_pct
0,l2_no_macro,11455,52.051401,5.611374e+06,4.341514e+07,72.992385
1,l1_l2_no_macro,11455,52.051403,5.611374e+06,4.341514e+07,72.992384
2,l1_no_macro,11455,53.516408,5.769309e+06,4.480756e+07,74.008812
3,baseline_no_macro,11455,53.516409,5.769309e+06,4.480756e+07,74.008814
4,l2_with_macro,11455,54.216860,5.844820e+06,4.344740e+07,82.888645
5,l1_l2_with_macro,11455,54.216860,5.844820e+06,4.344740e+07,82.888647
6,baseline_with_macro,11455,55.755036,6.010643e+06,4.482966e+07,79.388008
7,l1_with_macro,11455,55.755036,6.010643e+06,4.482966e+07,79.388008


## STEP 7. 설정 JSON 저장 및 결과 확인

In [7]:
config = {
    "dataset_version": "v002", "seed": SEED,
    "base_parameters": BASE_PARAMS,
    "regularization_grid": REGULARIZATIONS,
    "feature_columns_by_macro_variant": variants,
    "categorical_columns": categorical,
    "target": target,
    "train_target_period": manifest.get("train_period"),
    "validation_target_period": manifest.get("validation_period"),
    "validation_method": "origin_aligned_train_only_labels",
    "metric_priority": "WAPE_pct",
    "final_test_used": False,
    "validation_labels_used_for_fitting": False,
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
    "limitations": manifest.get("availability_limitation"),
    "note": "Train labels are capped at 2021Q4; validation labels are not added to later origins. This is NOT a fully expanding validation-label retraining evaluation."
}
(OUT / "experiment_config.json").write_text(
    json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
print("\n=== WAPE ranking (lower is better) ===")
print(summary.to_string(index=False))
print("\nSaved:", OUT)



=== WAPE ranking (lower is better) ===
      model_variant  rows  WAPE_pct      MAE_usd     RMSE_usd  sMAPE_pct
        l2_no_macro 11455 52.051401 5.611374e+06 4.341514e+07  72.992385
     l1_l2_no_macro 11455 52.051403 5.611374e+06 4.341514e+07  72.992384
        l1_no_macro 11455 53.516408 5.769309e+06 4.480756e+07  74.008812
  baseline_no_macro 11455 53.516409 5.769309e+06 4.480756e+07  74.008814
      l2_with_macro 11455 54.216860 5.844820e+06 4.344740e+07  82.888645
   l1_l2_with_macro 11455 54.216860 5.844820e+06 4.344740e+07  82.888647
baseline_with_macro 11455 55.755036 6.010643e+06 4.482966e+07  79.388008
      l1_with_macro 11455 55.755036 6.010643e+06 4.482966e+07  79.388008

Saved: D:\hjs\five-stack\artifacts\xgboost\v002\regularization


## STEP 8. 국가 · 제품군 · 분기 선택 예측 결과
아래 3개 값을 변경해 실행하세요. **검증 데이터에 이미 계산된 예측** 중 최저 WAPE 모델의 결과를 조회합니다. 같은 target 분기에 여러 origin/horizon이 있으면 여러 행이 표시됩니다. 2027 미래 예측 기능은 아닙니다.

In [8]:
COUNTRY = "USA"
HS_CODE = "330499"
TARGET_QUARTER = "2025Q4"

chosen = summary.iloc[0]["model_variant"]
example = predictions.loc[
    (predictions["model_variant"] == chosen)
    & (predictions["country_code"].astype(str).str.upper() == COUNTRY.upper())
    & (predictions["hs_code"].astype(str).str.replace(r"\.0$", "", regex=True) == HS_CODE)
    & (predictions["target_quarter"].astype(str) == TARGET_QUARTER)
].sort_values(["origin_quarter", "horizon_quarters"]).copy()

print("선택 모델:", chosen)
print("국가 / 제품군 / 대상 분기:", COUNTRY, HS_CODE, TARGET_QUARTER)
if example.empty:
    print("해당 조건의 검증 예측이 없습니다. validation_predictions.csv에서 가능한 조합을 확인하세요.")
else:
    example["absolute_error_usd"] = (example["actual_usd"] - example["predicted_usd"]).abs()
    display(example[["country_code", "hs_code", "origin_quarter", "target_quarter",
                     "horizon_quarters", "actual_usd", "predicted_usd", "absolute_error_usd"]])
    example.to_csv(OUT / "selected_example_predictions.csv", index=False, encoding="utf-8-sig")
    print("저장:", OUT / "selected_example_predictions.csv")

선택 모델: l2_no_macro
국가 / 제품군 / 대상 분기: USA 330499 2025Q4


,country_code,hs_code,origin_quarter,target_quarter,horizon_quarters,actual_usd,predicted_usd,absolute_error_usd
55722,USA,330499,2024Q3,2025Q4,5,359045142.0,604435392.0,245390250.0
56325,USA,330499,2024Q4,2025Q4,4,359045142.0,607106944.0,248061802.0
56782,USA,330499,2025Q1,2025Q4,3,359045142.0,941050624.0,582005482.0
57093,USA,330499,2025Q2,2025Q4,2,359045142.0,896754112.0,537708970.0
57257,USA,330499,2025Q3,2025Q4,1,359045142.0,949169536.0,590124394.0


저장: D:\hjs\five-stack\artifacts\xgboost\v002\regularization\selected_example_predictions.csv
